# Modeling Customer Churn: Putaran Pertama

Notebook ini membandingkan model pada data latih dengan parameter default. Model belum dipilih, dan data uji belum dipakai.

**Cara menguji:** model dipilih dari CV acak berstrata berulang. CV grup per `audit_group` hanya menjadi cek internal, dan skornya tidak masuk README atau laporan model.

**Metrik utama:** PR-AUC, dengan batas bawah `DummyClassifier`. ROC-AUC menjadi pendukung.

**Keterbatasan:** Analisis mengikuti dokumentasi UCI yang menyatakan tiap baris mewakili satu pelanggan. Audit menemukan pola berulang per 50 baris, sehingga independensi antarbaris tidak terjamin dan skor model bisa terlalu optimistis. Skor CV acak tidak boleh dibaca sebagai performa pada pelanggan baru.

## 1. Persiapan

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 180915
N_SPLITS = 5
N_REPEATS = 10

# Path relatif terhadap root project
ROOT = Path.cwd().parent if Path.cwd().name == 'program' else Path.cwd()

In [2]:
df = pd.read_csv(ROOT / 'dataset_clean' / 'customer_churn_clean.csv')
groups = pd.read_csv(ROOT / 'dataset_clean' / 'validation_groups.csv')['audit_group']
assert len(df) == len(groups)

TARGET = 'Churn'
FEATURES_ALL = [col for col in df.columns if col not in (TARGET, 'Age_Binning')]
FEATURES_REDUCED = [col for col in FEATURES_ALL if col not in ('Status', 'Complains')]
FEATURE_SETS = {'12 fitur': FEATURES_ALL, 'Tanpa Status dan Complains': FEATURES_REDUCED}

# audit_group disimpan terpisah, jadi tidak bisa terbawa menjadi fitur
assert 'audit_group' not in FEATURES_ALL and TARGET not in FEATURES_ALL
assert len(FEATURES_ALL) == 12 and len(FEATURES_REDUCED) == 10

print(f'Baris: {len(df):,} | Churn: {df[TARGET].sum():,} ({df[TARGET].mean():.2%})')
print('Fitur:', FEATURES_ALL)

Baris: 2,850 | Churn: 446 (15.65%)
Fitur: ['Call_Failure', 'Complains', 'Subscription_Length', 'Charge_Amount', 'Seconds_of_Use', 'Frequency_of_use', 'Frequency_of_SMS', 'Distinct_Called_Numbers', 'Age_Group', 'Tariff_Plan', 'Status', 'Customer_Value']


## 2. Memisahkan data uji

Sebanyak 20% baris dipisah acak berstrata. Baris uji tidak dipakai lagi di notebook ini.

In [3]:
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED
)
assert not set(train_idx) & set(test_idx)

train = df.loc[train_idx].reset_index(drop=True)
y_train = train[TARGET]
groups_train = groups.loc[train_idx].reset_index(drop=True)

print(f'Latih: {len(train):,} baris, {y_train.sum():,} churn ({y_train.mean():.2%})')
print(f'Uji  : {len(test_idx):,} baris, {df.loc[test_idx, TARGET].sum():,} churn')
print(f'Kelompok di data latih: {groups_train.nunique()}')

Latih: 2,280 baris, 357 churn (15.66%)
Uji  : 570 baris, 89 churn
Kelompok di data latih: 50


## 3. Model

Semua model memakai parameter default. Model yang peka skala diberi `StandardScaler` di dalam Pipeline, jadi scaler dipasang di dalam fold. Pembobotan kelas memakai `class_weight`. XGBoost memakai `scale_pos_weight`, yang dihitung dari data latih tiap fold. KNN tidak punya pembobotan kelas.

In [4]:
def scaled(model):
    return make_pipeline(StandardScaler(), model)


def xgb_model(**params):
    return XGBClassifier(random_state=SEED, eval_metric='logloss', **params)


# Nama konfigurasi: (model, varian penanganan)
CONFIGS = {
    'Dummy': (DummyClassifier(strategy='prior'), 'Tanpa'),
    'Logistic Regression': (scaled(LogisticRegression(max_iter=1000)), 'Tanpa'),
    'Logistic Regression + bobot': (scaled(LogisticRegression(max_iter=1000, class_weight='balanced')), 'Bobot'),
    'Decision Tree': (DecisionTreeClassifier(random_state=SEED), 'Tanpa'),
    'Decision Tree + bobot': (DecisionTreeClassifier(random_state=SEED, class_weight='balanced'), 'Bobot'),
    'KNN': (scaled(KNeighborsClassifier()), 'Tanpa'),
    'SVM RBF': (scaled(SVC()), 'Tanpa'),
    'SVM RBF + bobot': (scaled(SVC(class_weight='balanced')), 'Bobot'),
    'RandomForest': (RandomForestClassifier(random_state=SEED), 'Tanpa'),
    'RandomForest + bobot': (RandomForestClassifier(random_state=SEED, class_weight='balanced'), 'Bobot'),
    'XGBoost': (xgb_model(), 'Tanpa'),
    'XGBoost + bobot': (xgb_model(), 'Bobot XGB'),
}
print(f'Konfigurasi: {len(CONFIGS)}')

Konfigurasi: 12


## 4. Cara mengevaluasi

PR-AUC dan ROC-AUC dihitung di tiap fold, lalu dirata-ratakan per ulangan. Cara ini dipakai untuk kedua CV. Fold tanpa churn dilewati, karena kedua metrik tidak bisa dihitung di fold itu.

Skor dari prediksi gabungan semua fold tidak dipakai. Pada CV grup, porsi churn sangat berbeda antar-fold, sehingga skor antar-fold tidak setara, dan ROC-AUC `DummyClassifier` menjadi jauh di bawah 0,5.

Rentang min–maks menggambarkan variasi antarulangan, bukan interval kepercayaan.

In [5]:
def random_cv_repeats(X, y):
    splits = list(RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED
    ).split(X, y))
    return [splits[i:i + N_SPLITS] for i in range(0, len(splits), N_SPLITS)]


def group_cv_repeats(X, y, group_values):
    repeats = []
    for repeat in range(N_REPEATS):
        splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + repeat)
        splits = list(splitter.split(X, y, group_values))
        # Satu kelompok tidak boleh berada di data latih dan validasi sekaligus
        for fit_idx, val_idx in splits:
            assert not set(group_values.iloc[fit_idx]) & set(group_values.iloc[val_idx])
        repeats.append(splits)
    return repeats


def fit_and_score(model, variant, X_fit, y_fit, X_val):
    fitted = clone(model)
    if variant == 'Bobot XGB':
        # Dihitung di dalam fold, hanya dari data latih fold itu
        fitted.set_params(scale_pos_weight=(y_fit == 0).sum() / (y_fit == 1).sum())
    fitted.fit(X_fit, y_fit)
    if hasattr(fitted, 'predict_proba'):
        return fitted.predict_proba(X_val)[:, 1]
    return fitted.decision_function(X_val)


def evaluate(model, variant, X, y, repeats):
    rows = []
    for repeat, splits in enumerate(repeats):
        fold_pr, fold_roc = [], []
        for fit_idx, val_idx in splits:
            y_val = y.iloc[val_idx]
            # Fold tanpa churn dilewati, karena metrik tidak bisa dihitung
            if y_val.nunique() < 2:
                continue
            scores = fit_and_score(model, variant, X.iloc[fit_idx], y.iloc[fit_idx], X.iloc[val_idx])
            fold_pr.append(average_precision_score(y_val, scores))
            fold_roc.append(roc_auc_score(y_val, scores))
        rows.append({
            'repeat': repeat,
            'pr_auc': np.mean(fold_pr),
            'roc_auc': np.mean(fold_roc),
            'folds_skipped': N_SPLITS - len(fold_pr),
        })
    return rows

## 5. Menjalankan perbandingan

In [6]:
CV_REPEATS = {
    'CV acak': random_cv_repeats(train, y_train),
    'Cek grup': group_cv_repeats(train, y_train, groups_train),
}

started = time.time()
records = []
for feature_set, features in FEATURE_SETS.items():
    X_train = train[features]
    for cv_name, repeats in CV_REPEATS.items():
        for config_name, (model, variant) in CONFIGS.items():
            for row in evaluate(model, variant, X_train, y_train, repeats):
                records.append({'fitur': feature_set, 'cv': cv_name, 'model': config_name, **row})

results = pd.DataFrame(records)
print(f'Selesai dalam {(time.time() - started) / 60:.1f} menit')
print(f'Baris hasil: {len(results)} (24 konfigurasi x 2 CV x {N_REPEATS} ulangan)')

# Dihitung dari satu konfigurasi, karena fold-nya sama untuk semua model
one_config = results[(results['fitur'] == '12 fitur') & (results['model'] == 'Dummy')]
skipped = one_config.groupby('cv')['folds_skipped'].sum()
print(f"Fold tanpa churn yang dilewati: CV acak {skipped['CV acak']} dari {N_SPLITS * N_REPEATS}, "
      f"cek grup {skipped['Cek grup']} dari {N_SPLITS * N_REPEATS}")
assert skipped['CV acak'] == 0

Selesai dalam 5.3 menit
Baris hasil: 480 (24 konfigurasi x 2 CV x 10 ulangan)
Fold tanpa churn yang dilewati: CV acak 0 dari 50, cek grup 1 dari 50


## 6. Hasil

Format sel: rata-rata (min–maks antarulangan). Tabel diurutkan menurut PR-AUC CV acak.

In [7]:
def fmt(values):
    text = f'{values.mean():.3f} ({values.min():.3f}–{values.max():.3f})'
    return text.replace('.', ',')


def comparison_table(feature_set):
    subset = results[results['fitur'] == feature_set]
    table = pd.DataFrame({
        'PR-AUC CV acak': subset[subset['cv'] == 'CV acak'].groupby('model')['pr_auc'].apply(fmt),
        'ROC-AUC CV acak': subset[subset['cv'] == 'CV acak'].groupby('model')['roc_auc'].apply(fmt),
        'PR-AUC cek grup': subset[subset['cv'] == 'Cek grup'].groupby('model')['pr_auc'].apply(fmt),
        'ROC-AUC cek grup': subset[subset['cv'] == 'Cek grup'].groupby('model')['roc_auc'].apply(fmt),
    })
    order = (
        subset[subset['cv'] == 'CV acak'].groupby('model')['pr_auc'].mean()
        .sort_values(ascending=False).index
    )
    return table.loc[order]


print('=== 12 fitur ===')
print(comparison_table('12 fitur').to_string())

=== 12 fitur ===
                                  PR-AUC CV acak      ROC-AUC CV acak      PR-AUC cek grup     ROC-AUC cek grup
model                                                                                                          
XGBoost + bobot              0,932 (0,927–0,941)  0,980 (0,979–0,983)  0,476 (0,329–0,575)  0,801 (0,764–0,862)
XGBoost                      0,931 (0,920–0,939)  0,980 (0,977–0,983)  0,474 (0,323–0,588)  0,802 (0,754–0,858)
RandomForest                 0,914 (0,908–0,919)  0,978 (0,975–0,980)  0,504 (0,400–0,635)  0,826 (0,801–0,852)
RandomForest + bobot         0,909 (0,900–0,916)  0,977 (0,975–0,980)  0,483 (0,365–0,618)  0,827 (0,810–0,849)
KNN                          0,874 (0,858–0,883)  0,957 (0,948–0,960)  0,357 (0,284–0,435)  0,710 (0,596–0,745)
SVM RBF                      0,854 (0,847–0,863)  0,951 (0,946–0,957)  0,449 (0,341–0,527)  0,717 (0,678–0,747)
SVM RBF + bobot              0,830 (0,820–0,840)  0,963 (0,961–0,964)  0,432 (0,332–0,6

In [8]:
print('=== Tanpa Status dan Complains ===')
print(comparison_table('Tanpa Status dan Complains').to_string())

=== Tanpa Status dan Complains ===
                                  PR-AUC CV acak      ROC-AUC CV acak      PR-AUC cek grup     ROC-AUC cek grup
model                                                                                                          
XGBoost + bobot              0,889 (0,876–0,897)  0,970 (0,968–0,974)  0,228 (0,163–0,324)  0,611 (0,455–0,709)
XGBoost                      0,887 (0,878–0,896)  0,970 (0,968–0,974)  0,231 (0,171–0,294)  0,622 (0,481–0,720)
RandomForest                 0,826 (0,812–0,833)  0,965 (0,962–0,968)  0,223 (0,167–0,305)  0,622 (0,488–0,711)
RandomForest + bobot         0,818 (0,805–0,828)  0,965 (0,962–0,968)  0,232 (0,171–0,338)  0,632 (0,517–0,693)
KNN                          0,815 (0,805–0,822)  0,947 (0,943–0,953)  0,185 (0,157–0,226)  0,556 (0,468–0,614)
SVM RBF                      0,751 (0,742–0,757)  0,921 (0,917–0,925)  0,199 (0,164–0,239)  0,561 (0,499–0,642)
SVM RBF + bobot              0,733 (0,717–0,737)  0,944 (0,943–0,946)

## 7. Ketergantungan pada `Status` dan `Complains`

Selisih PR-AUC rata-rata antara 12 fitur dan 10 fitur. Ini melihat seberapa besar skor bergantung pada kedua kolom, bukan menguji leakage.

In [9]:
means = results.groupby(['cv', 'model', 'fitur'])['pr_auc'].mean().unstack('fitur')
means['selisih'] = means['12 fitur'] - means['Tanpa Status dan Complains']
dependence = means['selisih'].unstack('cv').round(3)
print(dependence.sort_values('CV acak', ascending=False).to_string())

cv                           CV acak  Cek grup
model                                         
Logistic Regression            0.225     0.256
Logistic Regression + bobot    0.224     0.234
Decision Tree                  0.114     0.120
SVM RBF                        0.103     0.250
SVM RBF + bobot                0.097     0.147
RandomForest + bobot           0.091     0.252
RandomForest                   0.088     0.282
Decision Tree + bobot          0.085     0.090
KNN                            0.059     0.172
XGBoost                        0.045     0.243
XGBoost + bobot                0.044     0.248
Dummy                          0.000     0.000


## 8. Pemeriksaan batas bawah

In [10]:
churn_rate = y_train.mean()
dummy = results[results['model'] == 'Dummy'].groupby('cv')[['pr_auc', 'roc_auc']].mean()
print(f'Churn rate data latih: {churn_rate:.4f}')
print(dummy.round(4).to_string())

# Model tanpa kemampuan harus mendapat ROC-AUC 0,5 pada kedua CV
assert (dummy['roc_auc'] - 0.5).abs().max() < 0.001
# PR-AUC cek grup hanya dicetak, karena mengikuti porsi churn tiap fold
assert abs(dummy.loc['CV acak', 'pr_auc'] - churn_rate) < 0.001

Churn rate data latih: 0.1566
          pr_auc  roc_auc
cv                       
CV acak   0.1566      0.5
Cek grup  0.1607      0.5


## Cara membaca hasil

- Skor memakai parameter default, jadi belum mewakili kemampuan terbaik tiap model.
- Logistic Regression hanya menerima fitur berupa angka, sehingga sulit menangkap pola naik lalu turun pada `Subscription_Length` dan `Age_Group`.
- Skor tinggi pada CV acak dibaca dengan curiga. Baris yang mirip bisa berada di data latih dan validasi sekaligus.
- Rentang cek grup lebar, dan tiap fold berbobot sama walaupun jumlah churn-nya sangat berbeda. Hasilnya sangat bergantung pada kelompok churn yang masuk tiap fold.
- Semua hubungan bersifat asosiatif, bukan kausal.
- Model belum dipilih di notebook ini.